# Weight Structure Profiling

Analyze weight matrix properties: spectral analysis, sparsity,
alignment between components, and norm distributions.

## Why This Matters

Weight structure examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_structure_profiling import (
    weight_spectral_profile, weight_sparsity_profile,
    weight_alignment_profile, weight_norm_distribution,
    weight_structure_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Spectral Profile

Condition number, effective rank, and top singular values.

In [ ]:
result = weight_spectral_profile(model, layer=0)
for name in ['W_Q', 'W_K', 'W_V', 'W_O', 'W_in', 'W_out']:
    info = result[name]
    print(f"  {name:5s}: cond={info['condition_number']:.1f} "
          f"rank={info['effective_rank']:.1f} "
          f"σ₁={info['spectral_norm']:.4f}")

## Sparsity Profile

In [ ]:
result = weight_sparsity_profile(model, layer=0)
for name in ['W_Q', 'W_K', 'W_V', 'W_O', 'W_in', 'W_out']:
    info = result[name]
    bar = '█' * int(info['sparsity'] * 30)
    print(f"  {name:5s}: sparsity={info['sparsity']:.3f} "
          f"mean={info['mean_abs_weight']:.4f} {bar}")

## Weight Alignment

In [ ]:
result = weight_alignment_profile(model, layer=0)
for pair in result['pairwise_alignments']:
    print(f"  {pair['matrix_a']} <-> {pair['matrix_b']}: "
          f"cos={pair['cosine_similarity']:.4f}")

## Norm Distribution Across Layers

In [ ]:
result = weight_norm_distribution(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn={p['total_attn_norm']:.3f} "
          f"mlp={p['total_mlp_norm']:.3f}")

## Structure Summary

In [ ]:
result = weight_structure_summary(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: W_Q cond={p['W_Q_condition']:.1f} "
          f"W_in cond={p['W_in_condition']:.1f}")